In [1]:
!git clone https://github.com/WildChlamydia/MiVOLO.git

Cloning into 'MiVOLO'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 257 (delta 63), reused 46 (delta 46), pack-reused 175 (from 1)
Receiving objects: 100% (257/257), 633.30 KiB | 2.55 MiB/s, done.
Resolving deltas: 100% (141/141), done.


In [2]:
!pip install -qr MiVOLO/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.2/699.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5

In [3]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.1.0
    Uninstalling ultralytics-8.1.0:
      Successfully uninstalled ultralytics-8.1.0


In [4]:
import os
import glob
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torchvision.ops import box_iou

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [25]:
keyframes_dir = '/root/.cache/kagglehub/datasets/khitrnhxun/aic-keyframesb1-reduced/versions/2/Keyframes'
all_keyframe_paths = dict()
for part in sorted(os.listdir(keyframes_dir)):
    data_part = part.split('_')[-1] # L01, L02 for ex
    all_keyframe_paths[data_part] =  dict()

for data_part in sorted(all_keyframe_paths.keys()):
    data_part_path = f'{keyframes_dir}/{data_part}'
    video_dirs = sorted(os.listdir(data_part_path))
    video_ids = [video_dir.split('_')[-1] for video_dir in video_dirs]
    for video_id, video_dir in zip(video_ids, video_dirs):
        keyframe_paths = sorted(glob.glob(f'{data_part_path}/{video_dir}/*.webp'))
        all_keyframe_paths[data_part][video_id] = keyframe_paths

# Init Model

In [5]:
!pip -q install gdown
!gdown --id 11i8pKctxz3wVkDBlWKvhYIh7kpVFXSZ4
!gdown --id 1CGNCkZQNj5WkP3rLpENWAOgrBQkUWRdw

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=11i8pKctxz3wVkDBlWKvhYIh7kpVFXSZ4
From (redirected): https://drive.google.com/uc?id=11i8pKctxz3wVkDBlWKvhYIh7kpVFXSZ4&confirm=t&uuid=2b2b83f9-b74e-4423-94bb-b02a60191a43
To: /content/model_imdb_cross_person_4.22_99.46.pth.tar
100% 110M/110M [00:01<00:00, 72.0MB/s]
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1CGNCkZQNj5WkP3rLpENWAOgrBQkUWRdw
From (redirected): https://drive.google.com/uc?id=1CGNCkZQNj5WkP3rLpENWAOgrBQkUWRdw&confirm=t&uuid=4677d532-d436-4ada-b0ce-53a879b852

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YOLO("yolo12x.pt")

In [21]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("khitrnhxun/aic-keyframesb1-reduced")

print("Path to dataset files:", path)

100%|██████████| 1.16G/1.16G [00:13<00:00, 90.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/khitrnhxun/aic-keyframesb1-reduced/versions/2


In [28]:
class VisualEncoding:

  def __init__(self,
    classes = ('person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic_light', 'fire_hydrant', 'stop_sign', 'parking_meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports_ball', 'kite', 'baseball_bat', 'baseball_glove', 'skateboard', 'surfboard', 'tennis_racket', 'bottle', 'wine_glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot_dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted_plant', 'bed', 'dining_table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell_phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy_bear', 'hair_drier', 'toothbrush'
              ),
    row_str = ["0", "1", "2", "3", "4", "5", "6"],
    col_str = ["a", "b", "c", "d", "e", "f", "g"]):

    self.classes = classes
    self.classes2idx = dict()
    for i, class_ in enumerate(classes):
        self.classes2idx[class_] = i
    self.n_row = len(row_str)
    self.n_col = len(col_str)

    x_pts = np.linspace(0, 1, self.n_row+1)
    y_pts = np.linspace(0, 1, self.n_col+1)

    self.grid_bboxes = []
    self.grid_labels = []
    for i in range(self.n_row):
        for j in range(self.n_col):
            label = col_str[j] + row_str[i]
            self.grid_bboxes.append([x_pts[j], y_pts[i], x_pts[j+1], y_pts[i+1]])
            self.grid_labels.append(label)

    self.grid_bboxes = np.array(self.grid_bboxes)

  def visualize_grid(self, grid_vis=None):
        if grid_vis is None:
            grid_vis = np.zeros((500, 500, 1))

        vis_h, vis_w, _ = grid_vis.shape
        font = cv2.FONT_HERSHEY_SIMPLEX
        fontScale = 0.5
        color = (255, 0, 0)
        thickness = 2
        for i in range(self.n_row*self.n_col):
            x_start, y_start, x_end, y_end = self.grid_bboxes[i]
            label = self.grid_labels[i]
            org = (int((x_start + (x_end-x_start)/2)*vis_w), int((y_start + (y_end-y_start)/2)*vis_h))

            # Draw text
            grid_vis = cv2.putText(grid_vis, label, org, font, fontScale, color, thickness, cv2.LINE_AA)
            # Draw grid
            grid_vis = cv2.rectangle(grid_vis, (int(x_start*vis_w), int(y_start*vis_h)), (int(x_end*vis_w), int(y_end*vis_h)), color, thickness)
        plt.imshow(grid_vis)

  def encode_bboxes(self, bboxes, labels):
        '''
        Args:
            bboxes: np.array: (n_bboxes, 4) - expected normalized bbox in form (x0, y0, x1, y1)
            labels: np.array: (n_bboxes, )
        '''
        iou = box_iou(torch.as_tensor(bboxes), torch.as_tensor(self.grid_bboxes))
        bboxes_idx, locs_idx = np.nonzero(iou.numpy())

        context = []
        for bbox_idx, loc_idx in zip(bboxes_idx, locs_idx):
            context.append(self.grid_labels[loc_idx] + self.classes[labels[bbox_idx]].replace(" ", ""))
        context = ' '.join(map(str, context))
        return context

  def encode_classes(self, labels):
        '''
        Args:
            labels: np.array: (n_bboxes, )
        '''
        unique_classes, counts = np.unique(labels, return_counts=True)
        context = []
        for unique_class, count in zip(unique_classes, counts):
            for i in range(count):
                context.append(self.classes[unique_class].replace(" ", "") + str(i))
        context = ' '.join(map(str, context))
        return context

  def encode_numbers(self, labels):
        '''
        Args:
            labels: np.array: (n_bboxes, )
        '''
        unique_classes, counts = np.unique(labels, return_counts=True)
        context = []
        for unique_class, count in zip(unique_classes, counts):
            context.append(self.classes[unique_class].replace(" ", "") + str(count))
        context = ' '.join(map(str, context))
        return context

  def encode(self, bboxes=None, labels=None, bboxes_colors=None, colors=None):
        '''
        Args:
            bboxes: np.array: (n_bboxes, 4) - expected normalized bbox in form (x0, y0, x1, y1)
            labels: np.array: (n_bboxes, )
        '''
        results = dict()
        if bboxes is not None:
            results['bbox'] = self.encode_bboxes(bboxes, labels)
            results['class'] = self.encode_classes(labels)
        else:
            results['bbox'] = results['class'] = None


        return results

In [38]:
import argparse
import logging
import os
import cv2
os.chdir("/content/MiVOLO")
import torch
import yt_dlp
from mivolo.data.data_reader import InputType, get_all_files, get_input_type
from mivolo.predictor import Predictor
from timm.utils import setup_default_logging
from types import SimpleNamespace

_logger = logging.getLogger("inference")

class MiVOLOPipeline:
  def __init__(self, detector_weights, checkpoint, device='cpu', verbose=False):
      self.config = SimpleNamespace(
          detector_weights=detector_weights,
          checkpoint=checkpoint,
          device=device,
          with_persons=True,
          disable_faces=False,
          draw=False
      )
      if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
      self.predictor = Predictor(self.config, verbose=verbose)

  def infer(self, inputs):
    age_gender = []
    for img in inputs:
      detected_objects, out_im = self.predictor.recognize(img)
      print(detected_objects.ages)
      print(detected_objects.genders)
      age_gender.append((detected_objects.ages[0], detected_objects.genders[0]))
    return age_gender

In [39]:
  # !pip -q install torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124

In [40]:
pipeline = MiVOLOPipeline(
    detector_weights="/content/yolov8x_person_face.pt",
    checkpoint="/content/model_imdb_cross_person_4.22_99.46.pth.tar",
    device="cuda:0"
    )

Model summary (fused): 112 layers, 68,125,494 parameters, 0 gradients, 257.4 GFLOPs


In [31]:
encoder = VisualEncoding()

In [33]:
import json

In [41]:
bs = 4

for key, video_keyframe_paths in tqdm(all_keyframe_paths.items()):
    os.makedirs(f"./context_encoded/json_encoded/{key}", exist_ok=True)
    video_ids = sorted(video_keyframe_paths.keys())

    for video_id in tqdm(video_ids):
        image_paths = video_keyframe_paths[video_id]
        video_result_dict = {}

        for i in range(0, len(image_paths), bs):
            batch_paths = image_paths[i:i+bs]
            results = model(batch_paths, conf=0.5, device=device, verbose=False)

            for img_path, result in zip(batch_paths, results):
                frame_name = os.path.basename(img_path)
                bboxes = result.boxes.xyxyn.cpu().numpy()
                labels = result.boxes.cls.cpu().numpy().astype(int)

                image = cv2.imread(img_path)
                H, W = image.shape[:2]

                if len(bboxes) == 0:
                    video_result_dict[frame_name] = {
                        "bbox": "",
                        "class": "",
                        "number": "",
                        # "attributes": []
                    }
                    continue

                encoded_bbox = encoder.encode_bboxes(bboxes, labels)
                encoded_class = encoder.encode_classes(labels)
                encoded_number = encoder.encode_numbers(labels)

                iou = box_iou(torch.as_tensor(bboxes), torch.as_tensor(encoder.grid_bboxes))
                bboxes_idx, locs_idx = np.nonzero(iou.numpy())

                attributes = []
                for bbox_idx, loc_idx in zip(bboxes_idx, locs_idx):
                    class_id = labels[bbox_idx]
                    class_name = encoder.classes[class_id]
                    grid = encoder.grid_labels[loc_idx]

                    attr = {"grid": grid, "class": class_name}
                    if class_name == "person":
                        x1, y1, x2, y2 = bboxes[bbox_idx]
                        x1, y1 = int(x1 * W), int(y1 * H)
                        x2, y2 = int(x2 * W), int(y2 * H)

                        cropped = image[y1:y2, x1:x2].copy()
                        results = pipeline.infer([cropped])
                        age, gender = results[0]
                        attr["age"] = age
                        attr["gender"] = gender
                    attributes.append(attr)

                video_result_dict[frame_name] = {
                    "bbox": encoded_bbox,
                    "class": encoded_class,
                    "number": encoded_number,
                    # "attributes": attributes
                }

        # Lưu file JSON
        save_path = f"./context_encoded/json_encoded/{key}/{video_id}.json"
        with open(save_path, "w") as f:
            json.dump(video_result_dict, f, indent=2)




  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/31 [00:00<?, ?it/s]

[35.83, 35.83]
['male', 'male']
[35.83, 35.83]
['male', 'male']
[35.83, 35.83]
['male', 'male']
[35.83, 35.83]
['male', 'male']
[35.83, 35.83]
['male', 'male']
[35.83, 35.83]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[31.61, 31.61]
['male', 'male']
[38.13, 38.13]
['female', 'female']
[38.13, 38.13]
['female', 'female']
[38.13, 38.13]
['female', 'female']
[38.13, 38.13]
['female', 'female']
[38.13, 38.13]
['female', 'female']
[38.13, 38.13]
['female', 'female']
[20.28, 20.28]
['male', 'male']
[20.28, 20.28]
['male', 'male']
[20.28, 20.28]
['male', 'male']
[20.28, 20.28]
['male', 'male']
[20.28, 20.28]
['male', 'male']
[20.28, 20.28]
['male', 'male']
[20.28, 20.28]
[

IndexError: list index out of range

In [23]:
!ls  /root/.cache/kagglehub/datasets/khitrnhxun/aic-keyframesb1-reduced/versions/2

Keyframes
